-- ==============================================================================
-- Silver Layer: Data Transformation & Standardization

-- Capa Silver: Transformación y estandarización de datos

-- Source Table: ecommerce_bronze.raw_sales_analytics

-- Objective: Cast types, standardise dates, and apply cleansing rules

-- Objetivo: Convertir tipos de datos, estandarizar fechas y aplicar reglas de limpieza.
-- ==============================================================================

In [0]:
-- 1. Create Target Schema
-- =============================================
CREATE SCHEMA IF NOT EXISTS ecommerce_silver;

In [0]:
-- 2. Create Target Delta Table with Constraints (if not exists)
CREATE TABLE IF NOT EXISTS ecommerce_silver.cleansed_sales (
    order_id INT,
    customer_id INT,
    order_date DATE,
    product_category STRING,
    region STRING,
    payment_method STRING,
    quantity INT,
    unit_price DECIMAL(10,2),
    discount DECIMAL(5,2),
    delivery_days INT,
    customer_rating DOUBLE,
    revenue DECIMAL(10,2),
    customer_key STRING,
    ingestion_timestamp TIMESTAMP
) USING DELTA;

ALTER TABLE ecommerce_silver.cleansed_sales ADD CONSTRAINT chk_silver_order_id CHECK (order_id IS NOT NULL);
ALTER TABLE ecommerce_silver.cleansed_sales ADD CONSTRAINT chk_silver_revenue CHECK (revenue >= 0);
ALTER TABLE ecommerce_silver.cleansed_sales ADD CONSTRAINT chk_silver_quantity CHECK (quantity > 0);

In [0]:
-- 3. Transform & Upsert Raw Data into Silver Delta Table (MERGE INTO)
-- Reemplazamos el CREATE OR REPLACE por lógica incremental segura
-- =============================================
MERGE INTO ecommerce_silver.cleansed_sales AS target
USING (
    SELECT DISTINCT
        CAST(order_id AS INT) AS order_id,
        CAST(customer_id AS INT) AS customer_id,
        TO_DATE(order_date, 'M/d/yyyy') AS order_date,
        TRIM(product_category) AS product_category,
        TRIM(region) AS region,
        TRIM(payment_method) AS payment_method,
        CAST(quantity AS INT) AS quantity,
        CAST(unit_price AS DECIMAL(10,2)) AS unit_price,
        CAST(discount AS DECIMAL(5,2)) AS discount,
        CAST(delivery_days AS INT) AS delivery_days,
        CAST(customer_rating AS DOUBLE) AS customer_rating,
        CAST(revenue AS DECIMAL(10,2)) AS revenue,
        MD5(CAST(customer_id AS STRING)) AS customer_key,
        CURRENT_TIMESTAMP() AS ingestion_timestamp
    FROM ecommerce_bronze.raw_sales
    WHERE order_id IS NOT NULL
) AS source

ON target.order_id = source.order_id
WHEN MATCHED THEN
  UPDATE SET 
    target.customer_id = source.customer_id,
    target.order_date = source.order_date,
    target.product_category = source.product_category,
    target.region = source.region,
    target.payment_method = source.payment_method,
    target.quantity = source.quantity,
    target.unit_price = source.unit_price,
    target.discount = source.discount,
    target.delivery_days = source.delivery_days,
    target.customer_rating = source.customer_rating,
    target.revenue = source.revenue,
    target.customer_key = source.customer_key,
    target.ingestion_timestamp = source.ingestion_timestamp
WHEN NOT MATCHED THEN
  INSERT (
    order_id, customer_id, order_date, product_category, region, 
    payment_method, quantity, unit_price, discount, delivery_days, 
    customer_rating, revenue, customer_key, ingestion_timestamp
  )
  VALUES (
    source.order_id, source.customer_id, source.order_date, source.product_category, source.region, 
    source.payment_method, source.quantity, source.unit_price, source.discount, source.delivery_days, 
    source.customer_rating, source.revenue, source.customer_key, source.ingestion_timestamp
  );